# 🧬 OO-Mamba — Training Pipeline (Local / VS Code)

**OO Mamba Latent Reasoning Engine** — inspiré de batteryphil's Mamba 2.8B.

Architecture:
- Base: `state-spaces/mamba-130m-hf` (130M params)
- Trainable: `x_proj`, `dt_proj`, `embed_tokens` seulement (~4%)
- Dark loops: tokens `=` pour raisonnement latent
- HaltingHead: MLP qui prédit P(halt) → arrête les boucles au bon moment

**Pipeline 4 phases:**
1. Phase 1 — Latent SFT (entraîne les SSM layers sur dark-loop data)
2. Phase 2 — HaltingHead (entraîne le contrôleur de boucles)
3. Phase 3 — Tool SFT (optionnel — entraîne les tool tokens)
4. Phase 4 — Merge + Export → `oo_mamba_v1.bin` pour bare-metal

⚠️ **Sans GPU**: Phase 1 dry-run OK, full training lent. Recommandé: Colab T4.

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────
import os, sys

# Chemin racine oo-model (notebooks/ est dans oo-model/notebooks/)
OO_MODEL_PATH = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
# Ou forcer:
# OO_MODEL_PATH = r'C:\Users\djibi\OneDrive\Bureau\baremetal\oo-model'

os.chdir(OO_MODEL_PATH)
if 'src' not in sys.path:
    sys.path.insert(0, os.path.join(OO_MODEL_PATH, 'src'))

import torch
print(f'CWD: {os.getcwd()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
else:
    print('CPU mode — dry-run OK, full training lent')

In [ ]:
# ── 1. Installer les dépendances Mamba ──────────────────────────────
# Mamba-ssm nécessite: transformers + mamba_ssm (CUDA) ou causal_conv1d
import subprocess

pkgs = ['transformers', 'accelerate', 'safetensors']
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    status = '✓' if r.returncode == 0 else '✗'
    print(f'{status} {pkg}')

# Pour GPU: installer aussi mamba-ssm (kernel CUDA officiel)
# !pip install mamba-ssm causal-conv1d
print('\nBase deps OK. Sur GPU Colab, décommenter la ligne mamba-ssm ci-dessus.')

In [ ]:
# ── 2. Vérifier le modèle base (téléchargement automatique HuggingFace) ──
# Premier run: télécharge ~260MB depuis HuggingFace (une seule fois)
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = 'state-spaces/mamba-130m-hf'

print(f'Chargement {BASE_MODEL}...')
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Test rapide sans charger les poids
print(f'Tokenizer OK: vocab={tok.vocab_size}')
sample = tok('Hello OO', return_tensors='pt')
print(f'Encode test: {sample["input_ids"].shape} tokens')

In [ ]:
# ── 3. Builder le dataset OO ────────────────────────────────────────
import json, subprocess

r = subprocess.run([sys.executable, 'scripts/build_dataset.py'],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('ERREUR:', r.stderr)

with open('data/processed/train.jsonl', encoding='utf-8') as f:
    rows = [json.loads(l) for l in f]
print(f'Dataset: {len(rows)} samples')

# Distribution domaines
from collections import Counter
domains = Counter(r['domain'] for r in rows)
for d, c in sorted(domains.items()):
    print(f'  {d}: {c}')

In [ ]:
# ── 4. Phase 1 — Dry-run (valide le pipeline, ~2min CPU) ───────────
r = subprocess.run(
    [sys.executable, 'scripts/train_latent.py',
     'configs/oo_v1_mamba_130m.json', '--dry-run'],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print('ERREUR:', r.stderr[-2000:])

In [ ]:
# ── 5. Phase 1 — Full SFT (décommenter pour lancer) ─────────────────
# ⚠️ CPU: ~2-4h  |  Colab T4: ~25min  |  Colab A100: ~8min

# r = subprocess.run(
#     [sys.executable, 'scripts/train_latent.py', 'configs/oo_v1_mamba_130m.json'],
#     capture_output=False
# )

print('Phase 1 full training commenté — décommenter pour lancer sur GPU')
print('Checkpoint sera sauvegardé dans: checkpoints/oo-mamba-phase1/')

In [ ]:
# ── 6. Phase 2 — HaltingHead (après Phase 1) ─────────────────────────
# Nécessite: checkpoints/oo-mamba-phase1/ existant
# Durée: ~10min CPU, ~2min GPU

# r = subprocess.run(
#     [sys.executable, 'scripts/train_halting_head.py', 'configs/oo_v1_mamba_130m.json'],
#     capture_output=False
# )

print('Phase 2 commenté — nécessite Phase 1 terminée d abord')
print('Checkpoint: checkpoints/oo-mamba-phase2-halt/halting_head.pt')

In [ ]:
# ── 7. Evaluation ─────────────────────────────────────────────────────
# Teste: domain probes, VRAM flatline, adaptive computation

from pathlib import Path

if not Path('checkpoints/oo-mamba-phase1').exists():
    print('Pas de checkpoint Phase 1 — lance le training d abord')
else:
    r = subprocess.run(
        [sys.executable, 'scripts/eval_mamba.py', 'configs/oo_v1_mamba_130m.json'],
        capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print('ERREUR:', r.stderr[-2000:])

In [ ]:
# ── 8. Phase 4 — Export binaire bare-metal ───────────────────────────
# Génère oo_mamba_v1.bin pour intégration dans llm-baremetal

# r = subprocess.run(
#     [sys.executable, 'scripts/export_ssm_binary.py',
#      'checkpoints/oo-mamba-phase1',
#      'checkpoints/oo_mamba_v1.bin'],
#     capture_output=False
# )

print('Export commenté — nécessite Phase 1 + Phase 2 terminées')
print('Output: checkpoints/oo_mamba_v1.bin (format OOSS bare-metal)')

## Résumé des checkpoints

| Phase | Script | Output | Durée GPU T4 |
|-------|--------|--------|-------------|
| 1 | `train_latent.py` | `checkpoints/oo-mamba-phase1/` | ~25min |
| 2 | `train_halting_head.py` | `checkpoints/oo-mamba-phase2-halt/halting_head.pt` | ~5min |
| eval | `eval_mamba.py` | rapport terminal | ~2min |
| export | `export_ssm_binary.py` | `checkpoints/oo_mamba_v1.bin` | <1min |

## Intégration dans llm-baremetal
```bash
cp checkpoints/oo_mamba_v1.bin ../llm-baremetal/models/
# Mettre à jour repl.cfg:
# model=oo_mamba_v1.bin
# model_type=ssm
```